# Notebook 03 — Eye Dataset Preparation

## Objective

Prepare a clean, leakage-safe dataset specifically for **Eye State Classification**.

The eye model will classify:

- Closed
- Open

This notebook will:

1. Load the leakage-safe split manifest.
2. Select only `Closed` and `Open`.
3. Verify train/validation/test counts.
4. Check image readability.
5. Analyze image dimensions.
6. Resize images to `224 × 224`.
7. Normalize pixel values during dataset preparation.
8. Apply training-only augmentation configuration.
9. Save a clean eye dataset manifest.
10. Create the folder structure needed for the eye models.



In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image
import shutil
import json

SEED = 42
IMG_SIZE = (224, 224)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

SPLIT_MANIFEST = PROJECT_ROOT / "outputs" / "dataset_split_manifest.csv"
EYE_OUTPUT_DIR = PROJECT_ROOT / "eye_dataset"
EYE_MANIFEST = PROJECT_ROOT / "outputs" / "eye_dataset_manifest.csv"

print("Project root:", PROJECT_ROOT)
print("Split manifest:", SPLIT_MANIFEST)
print("Eye dataset output:", EYE_OUTPUT_DIR)

if not SPLIT_MANIFEST.exists():
    raise FileNotFoundError(
        f"Missing {SPLIT_MANIFEST}. Run Notebook 02 first."
    )


Project root: d:\driver-drowsiness-detection
Split manifest: d:\driver-drowsiness-detection\outputs\dataset_split_manifest.csv
Eye dataset output: d:\driver-drowsiness-detection\eye_dataset


In [2]:
df = pd.read_csv(SPLIT_MANIFEST)

print("Manifest shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

display(df.head())


Manifest shape: (2900, 13)
Columns:
['filepath', 'filename', 'class', 'width', 'height', 'channels', 'mode', 'aspect_ratio', 'file_size_kb', 'valid', 'file_md5', 'similarity_group_id', 'split']


,filepath,filename,class,width,height,channels,mode,aspect_ratio,file_size_kb,valid,file_md5,similarity_group_id,split
0,D:\driver-drowsiness-detection\dataset\train\C...,_0.jpg,Closed,145,145,3,RGB,1.0000,6.09,True,f9319c997f4c75b111bd20bb6e22b1e1,SIM_0001,train
1,D:\driver-drowsiness-detection\dataset\train\C...,_1.jpg,Closed,142,135,3,RGB,1.0519,10.14,True,5e42467e4a1e07235cc3a0cdf7d32f01,SIM_0002,test
2,D:\driver-drowsiness-detection\dataset\train\C...,_10.jpg,Closed,131,127,3,RGB,1.0315,25.28,True,fbdbf633838b6268c8fc211258999021,SIM_0003,train
3,D:\driver-drowsiness-detection\dataset\train\C...,_100.jpg,Closed,288,287,3,RGB,1.0035,17.08,True,435eaa857612d1ae5454501c221abd5d,SIM_0004,train
4,D:\driver-drowsiness-detection\dataset\train\C...,_101.jpg,Closed,141,123,3,RGB,1.1463,26.18,True,9d0f18f02fc1990be2a0f9edb5376ff5,SIM_0005,test


## 1. Select the eye classes




In [3]:
EYE_CLASSES = ["Closed", "Open"]

eye_df = df[df["class"].isin(EYE_CLASSES)].copy()

if eye_df.empty:
    raise ValueError("No Closed/Open images found.")

print("Eye dataset images:", len(eye_df))
print("\nClass counts:")
print(eye_df["class"].value_counts().sort_index())

print("\nSplit counts:")
display(pd.crosstab(eye_df["split"], eye_df["class"]))


Eye dataset images: 1452

Class counts:
class
Closed    726
Open      726
Name: count, dtype: int64

Split counts:


class,Closed,Open
split,,
test,102,112
train,520,506
validation,104,108


## 2. Verify similarity-group leakage



In [4]:
group_split_counts = (
    eye_df.groupby("similarity_group_id")["split"]
          .nunique()
)

leaking_groups = group_split_counts[group_split_counts > 1]

print("Eye similarity groups appearing in multiple splits:", len(leaking_groups))

if len(leaking_groups) > 0:
    display(leaking_groups)
    raise AssertionError("Eye dataset leakage detected.")

print("✓ Eye dataset has no similarity-group leakage.")


Eye similarity groups appearing in multiple splits: 0
✓ Eye dataset has no similarity-group leakage.


## 3. Verify source images




In [5]:
bad_files = []
image_records = []

for _, row in eye_df.iterrows():
    path = Path(row["filepath"])

    if not path.exists():
        bad_files.append((str(path), "missing"))
        continue

    try:
        with Image.open(path) as img:
            img.verify()

        with Image.open(path) as img:
            width, height = img.size
            mode = img.mode

        image_records.append({
            "filepath": str(path),
            "class": row["class"],
            "split": row["split"],
            "similarity_group_id": row["similarity_group_id"],
            "width": width,
            "height": height,
            "mode": mode
        })

    except Exception as e:
        bad_files.append((str(path), str(e)))

print("Selected images:", len(eye_df))
print("Readable images:", len(image_records))
print("Bad/missing images:", len(bad_files))

if bad_files:
    display(pd.DataFrame(bad_files, columns=["filepath", "error"]).head(20))
    raise ValueError("One or more eye images could not be read.")

image_df = pd.DataFrame(image_records)
print("✓ All eye images are readable.")


Selected images: 1452
Readable images: 1452
Bad/missing images: 0
✓ All eye images are readable.


## 4. Analyze image dimensions




In [6]:
dimension_summary = (
    image_df.groupby("class")
             .agg(
                 images=("filepath", "size"),
                 unique_dimensions=("width", lambda x: len(set(zip(x, image_df.loc[x.index, "height"])))),
                 min_width=("width", "min"),
                 max_width=("width", "max"),
                 min_height=("height", "min"),
                 max_height=("height", "max"),
                 mean_width=("width", "mean"),
                 mean_height=("height", "mean")
             )
             .round(2)
)

display(dimension_summary)


,images,unique_dimensions,min_width,max_width,min_height,max_height,mean_width,mean_height
class,,,,,,,,
Closed,726,589,67,1625,54,1625,341.97,319.91
Open,726,578,50,1514,40,1515,340.00,307.68


## 5. Create the prepared eye dataset




In [7]:
# Create output folders
for split in ["train", "validation", "test"]:
    for cls in EYE_CLASSES:
        (EYE_OUTPUT_DIR / split / cls).mkdir(parents=True, exist_ok=True)

prepared_records = []

for _, row in eye_df.iterrows():
    src = Path(row["filepath"])
    split = row["split"]
    cls = row["class"]

    # Preserve original filename while placing it in its split/class folder.
    dst = EYE_OUTPUT_DIR / split / cls / src.name

    if not dst.exists():
        with Image.open(src) as img:
            # Convert to RGB so all images have a consistent 3-channel format.
            img = img.convert("RGB")
            img = img.resize(IMG_SIZE, Image.Resampling.LANCZOS)
            img.save(dst, quality=95)

    prepared_records.append({
        "original_filepath": str(src),
        "prepared_filepath": str(dst),
        "class": cls,
        "split": split,
        "similarity_group_id": row["similarity_group_id"],
        "original_width": row["width"],
        "original_height": row["height"],
        "prepared_width": IMG_SIZE[0],
        "prepared_height": IMG_SIZE[1]
    })

prepared_df = pd.DataFrame(prepared_records)

print("Prepared eye images:", len(prepared_df))
print("\nPrepared split counts:")
display(pd.crosstab(prepared_df["split"], prepared_df["class"]))


Prepared eye images: 1452

Prepared split counts:


class,Closed,Open
split,,
test,102,112
train,520,506
validation,104,108


## 6. Validate prepared images




In [8]:
validation_errors = []

for _, row in prepared_df.iterrows():
    path = Path(row["prepared_filepath"])

    if not path.exists():
        validation_errors.append((str(path), "missing"))
        continue

    try:
        with Image.open(path) as img:
            if img.size != IMG_SIZE:
                validation_errors.append((str(path), f"size={img.size}"))

            if img.mode != "RGB":
                validation_errors.append((str(path), f"mode={img.mode}"))

    except Exception as e:
        validation_errors.append((str(path), str(e)))

print("Validation errors:", len(validation_errors))

if validation_errors:
    display(pd.DataFrame(validation_errors, columns=["filepath", "error"]).head(20))
    raise AssertionError("Prepared eye dataset validation failed.")

print("✓ All prepared images are RGB and 224 × 224.")


Validation errors: 0
✓ All prepared images are RGB and 224 × 224.


## 7. Verify group integrity after preparation




In [9]:
prepared_group_splits = (
    prepared_df.groupby("similarity_group_id")["split"]
               .nunique()
)

prepared_leaks = prepared_group_splits[prepared_group_splits > 1]

print("Prepared groups appearing in multiple splits:", len(prepared_leaks))

if len(prepared_leaks) > 0:
    display(prepared_leaks)
    raise AssertionError("Prepared dataset contains leakage.")

print("✓ Prepared eye dataset remains leakage-safe.")


Prepared groups appearing in multiple splits: 0
✓ Prepared eye dataset remains leakage-safe.


## 8. Training augmentation configuration



In [10]:
augmentation_config = {
    "rotation": "small rotation",
    "zoom": "small zoom",
    "horizontal_flip": True,
    "brightness": "small brightness variation",
    "applies_to": "training only",
    "validation_augmented": False,
    "test_augmented": False
}

print(json.dumps(augmentation_config, indent=2))


{
  "rotation": "small rotation",
  "zoom": "small zoom",
  "horizontal_flip": true,
  "brightness": "small brightness variation",
  "applies_to": "training only",
  "validation_augmented": false,
  "test_augmented": false
}


## 9. Save the eye dataset manifest and summary


In [11]:
EYE_MANIFEST.parent.mkdir(parents=True, exist_ok=True)
prepared_df.to_csv(EYE_MANIFEST, index=False)

summary = {
    "task": "Eye State Classification",
    "classes": EYE_CLASSES,
    "total_images": int(len(prepared_df)),
    "image_size": list(IMG_SIZE),
    "splits": {
        split: int((prepared_df["split"] == split).sum())
        for split in ["train", "validation", "test"]
    },
    "class_counts": {
        cls: int((prepared_df["class"] == cls).sum())
        for cls in EYE_CLASSES
    },
    "similarity_group_leaks": int(len(prepared_leaks)),
    "augmentation": augmentation_config
}

summary_path = PROJECT_ROOT / "outputs" / "eye_dataset_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))

print("Saved:")
print(EYE_MANIFEST)
print(summary_path)


Saved:
d:\driver-drowsiness-detection\outputs\eye_dataset_manifest.csv
d:\driver-drowsiness-detection\outputs\eye_dataset_summary.json
